# TabPFN Deletion-Consistency Audit — Google Colab Notebook

**Thesis:** M.Tech CSE (Data Science), Integrated Programme, Oct 2027  
**Core question:** When a training row is deleted from a TabPFN pipeline, does the
resulting predictor behave identically to one that never saw the row?

---
## How to use this notebook
Run cells **one at a time, top to bottom**, using `Shift+Enter` or the ▶ button.
Each cell has a header explaining what it does.

> ⚠️ **Before starting:** Go to **Runtime → Change runtime type → T4 GPU**
> (TabPFN runs fine on CPU too, but T4 is faster)

---
**Differentiation note:** This audit differs from
[tabcustody](https://github.com/jrachid/tabcustody) (which does file-level scanning)
by measuring prediction-level metrics, calibrated probes, and statistical tests.

In [ ]:
# ===========================================================
# CELL 1: Check the environment
# ===========================================================
# Always run this first. It tells you:
#   - Which Python version Colab is using (should be 3.10 or 3.11)
#   - Whether a GPU is available
#   - The current date/time (for your lab notebook)
# ===========================================================

import sys
import datetime

print('Python version:', sys.version)
print('Run timestamp :', datetime.datetime.utcnow().isoformat(), 'UTC')
print()

# Check for GPU (optional — TabPFN also runs on CPU)
try:
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                             '--format=csv,noheader'],
                            capture_output=True, text=True)
    if result.returncode == 0:
        print('GPU detected:', result.stdout.strip())
    else:
        print('No GPU detected — running on CPU. This is fine for Iris dataset.')
except Exception:
    print('GPU check skipped.')

print()
print('RECORD THIS OUTPUT IN YOUR LAB NOTEBOOK before continuing.')

In [ ]:
# ===========================================================
# CELL 2: Install packages
# ===========================================================
# In Colab, `!` before a command runs it in the terminal
# (like opening Command Prompt inside the notebook).
#
# This installs:
#   tabpfn   — the model we are auditing
#   scipy    — for Wilcoxon statistical test
#
# scikit-learn, numpy, pandas are already in Colab by default.
#
# NOTE: tabpfn>=9.1 is REQUIRED (the old pin was tabpfn==2.0.3).
# The KV-cache experiment at the bottom uses
# fit_mode="fit_with_cache", which does not exist in tabpfn 2.x.
#
# IMPORTANT: tabpfn will download its pre-trained weights (~100 MB)
# the first time you call .fit(). This takes ~1 minute on Colab.
# ===========================================================

!pip install "tabpfn>=9.1,<10" scipy --quiet
print('Installation complete.')

In [ ]:
# ===========================================================
# CELL 3: Import all libraries and verify versions
# ===========================================================
# If any import fails, re-run Cell 2 first.
# ===========================================================

import hashlib
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Real TabPFN import
from tabpfn import TabPFNClassifier

warnings.filterwarnings('ignore')

# Print versions — RECORD ALL OF THESE in your lab notebook
import sklearn, scipy
import tabpfn
print('===== VERSION RECORD — copy to lab notebook =====')
print(f'tabpfn    : {tabpfn.__version__}')
print(f'sklearn   : {sklearn.__version__}')
print(f'numpy     : {np.__version__}')
print(f'pandas    : {pd.__version__}')
print(f'scipy     : {scipy.__version__}')
print('=================================================')

In [ ]:
# ===========================================================
# CELL 4: Configuration
# ===========================================================
# Change settings here, not scattered through the code.
# This is good practice — think of it like Java constants:
#   static final int N_TRIALS = 20;
# ===========================================================

# Random seed — MUST be recorded in thesis Methods section
RANDOM_SEED: int = 42

# How many rows to delete and test, one at a time
# Start with 20 for a pilot run; use 50+ for thesis-level evidence
N_DELETION_TRIALS: int = 20

# Fraction of data held out as test set
TEST_FRACTION: float = 0.25

# TabPFN settings
# n_estimators: number of forward passes (ensemble size)
# Higher = more accurate but slower. 4 is fast; 8 is balanced.
TABPFN_N_ESTIMATORS: int = 4

# Device: 'cuda' if GPU is available, 'cpu' otherwise
# TabPFN auto-detects, but set explicitly for reproducibility
try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

print(f'Device selected : {DEVICE}')
print(f'Random seed     : {RANDOM_SEED}  <- record this!')
print(f'Deletion trials : {N_DELETION_TRIALS}')
print(f'TabPFN ensemble : {TABPFN_N_ESTIMATORS} estimators')

In [ ]:
# ===========================================================
# CELL 5: Helper functions
# ===========================================================
# Small reusable utilities — same as the local prototype.
# ===========================================================

def set_global_seed(seed: int) -> None:
    """Lock all random number generators to a fixed seed."""
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except ImportError:
        pass


def hash_dataframe(df: pd.DataFrame) -> str:
    """
    MD5 fingerprint of a DataFrame.
    Record this in your lab notebook — it proves you used the exact same data.
    """
    df_bytes = pd.util.hash_pandas_object(df, index=True).values.tobytes()
    return hashlib.md5(df_bytes).hexdigest()


def max_abs_diff(probs_A: np.ndarray, probs_B: np.ndarray) -> float:
    """
    Maximum Absolute Difference between two probability matrices.
    0 = identical, higher = more drift between Path A and Path B.
    """
    return float(np.max(np.abs(probs_A - probs_B)))


def mean_l2_distance(probs_A: np.ndarray, probs_B: np.ndarray) -> float:
    """
    Average Euclidean distance between probability vectors, per test row.
    Lower = Path A and Path B agree more on average.
    """
    per_row = np.linalg.norm(probs_A - probs_B, axis=1)
    return float(np.mean(per_row))


def print_section(title: str) -> None:
    print('\n' + '=' * 60)
    print(f'  {title}')
    print('=' * 60)


print('Helper functions defined OK.')

In [ ]:
# ===========================================================
# CELL 6: Load dataset and split
# ===========================================================
# Using Iris for the pilot run (150 rows, 4 features, 3 classes).
# This is small enough to run in seconds on CPU.
#
# To use your own dataset later, replace the load_iris() block
# with:  df = pd.read_csv('your_file.csv')
# ===========================================================

set_global_seed(RANDOM_SEED)

# Load dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

dataset_hash = hash_dataframe(X)

print_section('DATASET')
print(f'Name         : Iris (sklearn built-in)')
print(f'Hash         : {dataset_hash}  <- record this!')
print(f'Total rows   : {len(X)}')
print(f'Features     : {list(X.columns)}')
print(f'Classes      : {np.unique(y).tolist()}  (0=setosa, 1=versicolor, 2=virginica)')
print()

# Train / test split
# stratify=y ensures each class has the same proportion in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_SEED,
    stratify=y
)

# Reset integer row indices after splitting
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f'Train rows   : {len(X_train)}')
print(f'Test rows    : {len(X_test)}')

# Fit the "stale" scaler on the full training set (before any deletion)
# Path A reuses this; Path B ignores it and builds fresh
stale_scaler = StandardScaler()
stale_scaler.fit(X_train)
print(f'Stale scaler fitted on {len(X_train)} rows.')

In [ ]:
# ===========================================================
# CELL 7: Define the two deletion paths (real TabPFN)
# ===========================================================
# This is the heart of the audit.
#
# PATH A: Naive delete — reuses stale scaler (the "bug")
# PATH B: Clean rebuild — refit scaler from scratch (correct)
#
# IMPORTANT: TabPFN downloads its weights (~100 MB) on the first
# call to .fit(). You will see a progress bar. This is normal.
# ===========================================================

def build_tabpfn() -> TabPFNClassifier:
    """
    Build a fresh TabPFN classifier with fixed settings.
    Called inside each deletion path so each trial gets a fresh model.

    seed=RANDOM_SEED ensures the same ensemble configuration each time.
    """
    return TabPFNClassifier(
        n_estimators=TABPFN_N_ESTIMATORS,
        device=DEVICE,
        random_state=RANDOM_SEED,
    )


def path_A_naive_delete(
    X_tr, y_tr, X_te, delete_idx, stale_sc
) -> np.ndarray:
    """
    PATH A: Naive delete (stale scaler).

    The scaler was fitted on the FULL training set, including the
    row being deleted. After deletion, we still use those stale
    statistics. This is the 'bug' we are auditing.

    Returns: probability matrix, shape (n_test, n_classes)
    """
    # 1. Remove the target row
    X_minus = X_tr.drop(index=delete_idx).reset_index(drop=True)
    y_minus = np.delete(y_tr, X_tr.index.get_loc(delete_idx))

    # 2. Scale with the STALE scaler (still encodes deleted row)
    X_minus_sc = stale_sc.transform(X_minus)
    X_test_sc  = stale_sc.transform(X_te)

    # 3. Train real TabPFN and get prediction probabilities
    model = build_tabpfn()
    model.fit(X_minus_sc, y_minus)
    return model.predict_proba(X_test_sc)


def path_B_clean_rebuild(
    X_tr, y_tr, X_te, delete_idx
) -> np.ndarray:
    """
    PATH B: Clean rebuild (fresh scaler).

    The row is removed FIRST, then a fresh scaler is fitted on the
    reduced training set. The deleted row's values never influence
    the scaler's mean or std.

    Returns: probability matrix, shape (n_test, n_classes)
    """
    # 1. Remove the target row
    X_minus = X_tr.drop(index=delete_idx).reset_index(drop=True)
    y_minus = np.delete(y_tr, X_tr.index.get_loc(delete_idx))

    # 2. Fit a FRESH scaler — no memory of the deleted row
    fresh_sc = StandardScaler()
    fresh_sc.fit(X_minus)   # <-- key difference from Path A

    # 3. Scale with the fresh scaler
    X_minus_sc = fresh_sc.transform(X_minus)
    X_test_sc  = fresh_sc.transform(X_te)

    # 4. Train real TabPFN and get prediction probabilities
    model = build_tabpfn()
    model.fit(X_minus_sc, y_minus)
    return model.predict_proba(X_test_sc)


print('Path A and Path B functions defined.')
print()
print('NOTE: First call to TabPFN will download model weights (~100 MB).')
print('      Subsequent calls are fast (weights are cached).')

In [ ]:
# ===========================================================
# CELL 8: Blind baseline (new — not in local prototype)
# ===========================================================
# WHAT IS A BLIND BASELINE?
# A blind baseline runs the same audit on a model that has NO
# real signal — the labels are randomly shuffled, so the model
# cannot learn anything meaningful.
#
# WHY IS IT NEEDED?
# Without it, we cannot tell whether Path A vs Path B differences
# are due to the stale scaler, or just due to natural variance in
# TabPFN's predictions when training data changes by one row.
#
# HOW TO INTERPRET:
# Report excess = (real MAD) - (blind baseline MAD)
# If excess ≈ 0, the effect is just noise, not stale preprocessing.
# If excess > 0, the stale scaler genuinely causes extra drift.
#
# SOURCE: Wahdany et al., 2026, arXiv:2610.24278
# ===========================================================

print('Computing blind baseline...')
print('(This shuffles labels so the model learns nothing real.)')
print()

rng_blind = np.random.default_rng(seed=RANDOM_SEED + 1000)  # Different seed from main audit
y_train_shuffled = rng_blind.permutation(y_train)           # Random labels = no real signal

# Run 5 trials on the shuffled dataset to estimate the noise floor
BLIND_TRIALS = 5
blind_delete_indices = np.random.default_rng(RANDOM_SEED).choice(
    X_train.index.tolist(), size=BLIND_TRIALS, replace=False
)

blind_mad_scores = []

for i, del_idx in enumerate(blind_delete_indices, 1):
    # Path A on shuffled labels
    probs_A_blind = path_A_naive_delete(
        X_train, y_train_shuffled, X_test, del_idx, stale_scaler
    )
    # Path B on shuffled labels
    probs_B_blind = path_B_clean_rebuild(
        X_train, y_train_shuffled, X_test, del_idx
    )
    mad_blind = max_abs_diff(probs_A_blind, probs_B_blind)
    blind_mad_scores.append(mad_blind)
    print(f'  Blind trial {i} | row={del_idx} | MAD={mad_blind:.5f}')

blind_mad_arr = np.array(blind_mad_scores)
BLIND_BASELINE_MEAN = blind_mad_arr.mean()
BLIND_BASELINE_STD  = blind_mad_arr.std()

print()
print(f'Blind baseline MAD — mean: {BLIND_BASELINE_MEAN:.5f}  std: {BLIND_BASELINE_STD:.5f}')
print('This is the noise floor: drift expected even when nothing real was learned.')

In [ ]:
# ===========================================================
# CELL 9: Main deletion audit loop (real TabPFN)
# ===========================================================
# This is the main experiment.
# For each of N_DELETION_TRIALS random rows:
#   1. Run Path A (naive delete, stale scaler)
#   2. Run Path B (clean rebuild, fresh scaler)
#   3. Record MAD, L2 distance, and wall-clock time
#
# EXPECTED RUNTIME: ~5–15 minutes on Colab T4 for 20 trials
#                   TabPFN is fast after first weight download
# ===========================================================

set_global_seed(RANDOM_SEED)

# Choose rows to delete (same logic as local prototype, same seed = same rows)
rng = np.random.default_rng(seed=RANDOM_SEED)
delete_indices = rng.choice(
    X_train.index.tolist(),
    size=N_DELETION_TRIALS,
    replace=False
)

print_section(f'RUNNING {N_DELETION_TRIALS} DELETION TRIALS — REAL TabPFN')
print(f'Rows selected: {delete_indices.tolist()}')
print()

mad_scores = []
l2_scores  = []
times_A    = []
times_B    = []

for trial_num, del_idx in enumerate(delete_indices, start=1):

    # --- PATH A ---
    t0 = time.perf_counter()
    probs_A = path_A_naive_delete(
        X_train, y_train, X_test, del_idx, stale_scaler
    )
    times_A.append(time.perf_counter() - t0)

    # --- PATH B ---
    t0 = time.perf_counter()
    probs_B = path_B_clean_rebuild(
        X_train, y_train, X_test, del_idx
    )
    times_B.append(time.perf_counter() - t0)

    # --- Metrics ---
    mad = max_abs_diff(probs_A, probs_B)
    l2  = mean_l2_distance(probs_A, probs_B)
    mad_scores.append(mad)
    l2_scores.append(l2)

    print(f'  Trial {trial_num:2d} | row={del_idx:3d} | '
          f'MAD={mad:.5f} | L2={l2:.5f} | '
          f't_A={times_A[-1]:.2f}s | t_B={times_B[-1]:.2f}s')

print()
print('All trials complete.')

In [ ]:
# ===========================================================
# CELL 10: Aggregate results and statistical test
# ===========================================================

mad_arr = np.array(mad_scores)
l2_arr  = np.array(l2_scores)

print_section('AGGREGATE RESULTS')

print('Max Absolute Difference (MAD):')
print(f'  Mean  = {mad_arr.mean():.5f}')
print(f'  Std   = {mad_arr.std():.5f}')
print(f'  Max   = {mad_arr.max():.5f}')
print(f'  Min   = {mad_arr.min():.5f}')

print(f'\nMean L2 Distance:')
print(f'  Mean  = {l2_arr.mean():.5f}')
print(f'  Std   = {l2_arr.std():.5f}')

# --- Excess over blind baseline ---
# This is the key number: how much MORE drift does the stale scaler
# cause, compared to the blind noise floor?
excess_mad = mad_arr.mean() - BLIND_BASELINE_MEAN
print(f'\nBlind baseline MAD (noise floor): {BLIND_BASELINE_MEAN:.5f}')
print(f'Excess MAD (real - blind)        : {excess_mad:.5f}')
print('(Positive excess = stale scaler causes real drift above noise)')

print(f'\nWall-clock time (unlearning cost):')
print(f'  Path A mean = {np.mean(times_A):.3f}s')
print(f'  Path B mean = {np.mean(times_B):.3f}s')
print(f'  Overhead of clean rebuild = {np.mean(times_B) - np.mean(times_A):+.3f}s')

print_section('STATISTICAL TEST')

if np.all(mad_arr == mad_arr[0]):   # All identical — no variance to test
    print('All MAD values are identical. Wilcoxon test skipped.')
else:
    # Wilcoxon signed-rank test: is MAD significantly greater than 0?
    # H0: Path A == Path B (MAD = 0)
    # H1: Path A != Path B (MAD > 0, one-sided)
    stat, p_val = stats.wilcoxon(mad_arr, alternative='greater')
    # Note: p shown as 0.0000 means p < 0.0001 — always report it that way
    p_display = '< 0.0001' if p_val < 0.0001 else f'= {p_val:.4f}'
    print(f'Wilcoxon signed-rank test (H1: MAD > 0)')
    print(f'  Statistic = {stat:.4f}')
    print(f'  p-value   {p_display}')
    if p_val < 0.05:
        print(f'  → SIGNIFICANT at α=0.05')
        print(f'    Mechanism isolated: stale StandardScaler statistics cause')
        print(f'    measurable prediction drift in real TabPFN.')
    else:
        print(f'  → NOT significant at α=0.05')
        print(f'    No detectable drift from stale preprocessing in this run.')

In [ ]:
# ===========================================================
# CELL 11: Plot the results
# ===========================================================
# A visual check is always useful. This creates two plots:
#   Left : MAD per trial (bar chart)
#   Right: MAD distribution (histogram) with blind baseline marked
# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('TabPFN Deletion Audit — Path A vs Path B (MAD)', fontsize=13)

# Left: per-trial bar chart
ax = axes[0]
ax.bar(range(1, N_DELETION_TRIALS + 1), mad_scores, color='steelblue', alpha=0.8)
ax.axhline(BLIND_BASELINE_MEAN, color='red', linestyle='--', label='Blind baseline mean')
ax.set_xlabel('Trial number')
ax.set_ylabel('Max Absolute Difference (MAD)')
ax.set_title('MAD per deletion trial')
ax.legend()

# Right: histogram
ax = axes[1]
ax.hist(mad_scores, bins=10, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(BLIND_BASELINE_MEAN, color='red', linestyle='--', label='Blind baseline mean')
ax.axvline(mad_arr.mean(), color='orange', linestyle='-', label='Real MAD mean')
ax.set_xlabel('MAD')
ax.set_ylabel('Count')
ax.set_title('Distribution of MAD scores')
ax.legend()

plt.tight_layout()
plt.savefig('audit_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to audit_results.png')

In [ ]:
# ===========================================================
# CELL 12: Save results and download
# ===========================================================
# This saves a text summary you can paste into your lab notebook.
# It also downloads the plot image to your local computer.
# ===========================================================

import json, tabpfn, sklearn, scipy as sp

# Build a results dictionary
results = {
    'run_metadata': {
        'timestamp_utc'       : datetime.datetime.utcnow().isoformat(),
        'dataset'             : 'Iris (sklearn built-in)',
        'dataset_hash'        : dataset_hash,
        'random_seed'         : RANDOM_SEED,
        'n_deletion_trials'   : N_DELETION_TRIALS,
        'deleted_row_indices' : delete_indices.tolist(),
        'tabpfn_version'      : tabpfn.__version__,
        'sklearn_version'     : sklearn.__version__,
        'numpy_version'       : np.__version__,
        'pandas_version'      : pd.__version__,
        'scipy_version'       : sp.__version__,
        'device'              : DEVICE,
        'tabpfn_n_estimators' : TABPFN_N_ESTIMATORS,
    },
    'per_trial': {
        'deleted_indices': delete_indices.tolist(),
        'mad_scores'     : mad_scores,
        'l2_scores'      : l2_scores,
        'times_A_s'      : times_A,
        'times_B_s'      : times_B,
    },
    'aggregate': {
        'mad_mean'            : float(mad_arr.mean()),
        'mad_std'             : float(mad_arr.std()),
        'mad_max'             : float(mad_arr.max()),
        'mad_min'             : float(mad_arr.min()),
        'l2_mean'             : float(l2_arr.mean()),
        'l2_std'              : float(l2_arr.std()),
        'blind_baseline_mean' : float(BLIND_BASELINE_MEAN),
        'blind_baseline_std'  : float(BLIND_BASELINE_STD),
        'excess_mad'          : float(excess_mad),
        'path_A_time_mean_s'  : float(np.mean(times_A)),
        'path_B_time_mean_s'  : float(np.mean(times_B)),
    },
    'caveats': [
        'Mechanism isolated: stale StandardScaler statistics only.',
        'Does NOT prove TabPFN internally memorises rows.',
        'Pilot run: 20 trials, 1 seed. Use >=50 trials x 3 seeds for thesis.',
        'Blind baseline used shuffled labels (not a never-trained model).',
        'Differentiate from tabcustody (github.com/jrachid/tabcustody).',
    ]
}

# Save to JSON
with open('audit_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to audit_results.json')
print('Plot saved to audit_results.png')
print()
print('To download both files to your computer:')
print('  - Click the folder icon (📁) in the left sidebar')
print('  - Right-click audit_results.json → Download')
print('  - Right-click audit_results.png  → Download')
print()
print('Save both to: c:\\Users\\Omni\\TabPFN-Deletion-Audit\\results\\')

---
## Experiment 2 - KV cache (`fit_with_cache`) audit

`tabpfn>=9.1` can precompute the training set's attention state once and reuse it
(`fit_mode="fit_with_cache"`). That cache is computed **from the training rows**, so:
does a deleted row leave a larger trace when the pipeline uses the cache?

1. **Sanity check:** normal fit vs cache fit on identical data must give (near-)identical
   predictions. If not, the fit mode itself adds noise and we need to know that first.
2. **Footprint:** for each deleted row, refit both modes on the reduced data (clean rebuild)
   and measure how much test predictions move vs the full-data fit (MAD).
3. **Probe:** after each refit, predict the deleted row and record the confidence on its
   true class. High residual confidence on a row the model never saw is a memorization signal.

Also recorded: refit wall-clock time per mode (the unlearning cost).
Saves `fit_cache_results.json`. Standalone version: `fit_cache_audit.py`.


In [ ]:
# ===========================================================
# CELL: KV-cache audit - setup, sanity check, full-data fits
# ===========================================================
# Reuses: X_train, X_test, y_train, RANDOM_SEED, DEVICE,
#         TABPFN_N_ESTIMATORS, print_section from earlier cells.
# ===========================================================

MODE_NORMAL = 'auto'
MODE_CACHE  = 'fit_with_cache'

def build_tabpfn_mode(fit_mode: str) -> TabPFNClassifier:
    """Fresh TabPFN in the given fit mode (fresh instance = no state leaks)."""
    return TabPFNClassifier(
        n_estimators=TABPFN_N_ESTIMATORS,
        device=DEVICE,
        random_state=RANDOM_SEED,
        fit_mode=fit_mode,
    )

def max_abs_diff2(A, B):
    """Worst-case disagreement between two probability matrices."""
    return float(np.max(np.abs(A - B)))

print_section('CACHE AUDIT - SANITY CHECK')

# Fresh scaler on the FULL training set (both modes see identical data)
full_scaler = StandardScaler().fit(X_train)
X_train_sc = full_scaler.transform(X_train)
X_test_sc  = full_scaler.transform(X_test)

t0 = time.perf_counter()
m_normal_full = build_tabpfn_mode(MODE_NORMAL); m_normal_full.fit(X_train_sc, y_train)
t_fit_normal_full = time.perf_counter() - t0

t0 = time.perf_counter()
m_cache_full = build_tabpfn_mode(MODE_CACHE); m_cache_full.fit(X_train_sc, y_train)
t_fit_cache_full = time.perf_counter() - t0

P_full_normal = m_normal_full.predict_proba(X_test_sc)
P_full_cache  = m_cache_full.predict_proba(X_test_sc)

sanity_mad = max_abs_diff2(P_full_normal, P_full_cache)
print(f'MAD(normal fit, cache fit) on identical data = {sanity_mad:.6f}')
print(f'Fit time: normal {t_fit_normal_full:.2f}s | cache {t_fit_cache_full:.2f}s')
print('(should be ~0; if not, fit mode itself adds noise)')


In [ ]:
# ===========================================================
# CELL: KV-cache audit - deletion footprint + deleted-row probe
# ===========================================================
# For each deleted row (same delete_indices as the main audit):
#   clean rebuild (fresh scaler) in BOTH fit modes
#   footprint  = MAD(full-data predictions, deleted-data predictions)
#   probe      = P(true class) of the deleted row after the refit
# EXPECTED RUNTIME: ~10-25 min on T4 (2 refits per trial)
# ===========================================================

set_global_seed(RANDOM_SEED)
print_section('CACHE AUDIT - DELETION TRIALS')

footprint_normal, footprint_cache = [], []
probe_normal, probe_cache = [], []
refit_t_normal, refit_t_cache = [], []

for trial_num, del_idx in enumerate(delete_indices, start=1):

    deleted_row   = X_train.iloc[[del_idx]]      # keep for the probe
    deleted_label = int(y_train[del_idx])

    # Clean rebuild: remove row, FRESH scaler on the rest
    X_minus = X_train.drop(index=del_idx).reset_index(drop=True)
    y_minus = np.delete(y_train, del_idx)
    fresh_sc = StandardScaler().fit(X_minus)
    X_minus_sc      = fresh_sc.transform(X_minus)
    X_test_sc_fresh = fresh_sc.transform(X_test)
    deleted_row_sc  = fresh_sc.transform(deleted_row)

    # Normal mode refit
    t0 = time.perf_counter()
    m_n = build_tabpfn_mode(MODE_NORMAL); m_n.fit(X_minus_sc, y_minus)
    refit_t_normal.append(time.perf_counter() - t0)
    P_del_n = m_n.predict_proba(X_test_sc_fresh)
    probe_normal.append(float(m_n.predict_proba(deleted_row_sc)[0, deleted_label]))

    # Cache mode refit
    t0 = time.perf_counter()
    m_c = build_tabpfn_mode(MODE_CACHE); m_c.fit(X_minus_sc, y_minus)
    refit_t_cache.append(time.perf_counter() - t0)
    P_del_c = m_c.predict_proba(X_test_sc_fresh)
    probe_cache.append(float(m_c.predict_proba(deleted_row_sc)[0, deleted_label]))

    footprint_normal.append(max_abs_diff2(P_full_normal, P_del_n))
    footprint_cache.append(max_abs_diff2(P_full_cache,  P_del_c))

    print(f'  Trial {trial_num:2d} | row={del_idx:3d} | '
          f'footprint N={footprint_normal[-1]:.5f} C={footprint_cache[-1]:.5f} | '
          f'probe N={probe_normal[-1]:.3f} C={probe_cache[-1]:.3f}')

# --- Aggregate + statistics ---
fp_n = np.array(footprint_normal); fp_c = np.array(footprint_cache)
pb_n = np.array(probe_normal);     pb_c = np.array(probe_cache)

print_section('CACHE AUDIT - RESULTS')
print(f'Footprint : normal mean={fp_n.mean():.5f} | cache mean={fp_c.mean():.5f}')
print(f'Probe     : normal mean={pb_n.mean():.3f} | cache mean={pb_c.mean():.3f} (chance=0.333)')
print(f'Refit time: normal {np.mean(refit_t_normal):.2f}s | cache {np.mean(refit_t_cache):.2f}s')

diff = fp_c - fp_n
if np.all(diff == 0.0):
    p_modes = 1.0
    print('Footprints identical across modes - Wilcoxon skipped.')
else:
    stat_c, p_modes = stats.wilcoxon(diff, alternative='greater')
    print(f'Wilcoxon (cache footprint > normal): stat={stat_c:.4f} p={p_modes:.4f}')

cache_results = {
    'sanity_mad_same_data': sanity_mad,
    'footprint_normal': footprint_normal, 'footprint_cache': footprint_cache,
    'probe_normal': probe_normal, 'probe_cache': probe_cache,
    'footprint_normal_mean': float(fp_n.mean()), 'footprint_cache_mean': float(fp_c.mean()),
    'probe_normal_mean': float(pb_n.mean()), 'probe_cache_mean': float(pb_c.mean()),
    'refit_time_normal_mean_s': float(np.mean(refit_t_normal)),
    'refit_time_cache_mean_s': float(np.mean(refit_t_cache)),
    'wilcoxon_cache_vs_normal_p': float(p_modes),
    'caveats': [
        'Footprint includes the fresh-scaler difference (Path B convention).',
        'Probe is exploratory; high P(true class) is a signal, not proof.',
        'Pilot run: 20 trials, 1 seed, Iris. Thesis needs 50+ x 3 seeds, real datasets.',
    ],
}
with open('fit_cache_results.json', 'w') as f:
    json.dump(cache_results, f, indent=2)
print('Saved fit_cache_results.json - download it into results/ like the others.')
print('Copy the numbers above into lab_notebook.txt as RUN 003.')


## Mandatory caveats before interpreting any output

1. **Mechanism isolated:** Any A-vs-B difference isolates *only* the effect of reusing stale `StandardScaler` statistics. It does **not** prove that TabPFN internally memorises rows.

2. **Novelty statement:** We did not find a published study doing this exact audit. This does not mean it is the first; it means our search did not find one. Always differentiate from [tabcustody](https://github.com/jrachid/tabcustody), which does file-level scanning.

3. **Sample size:** 20 trials × 1 seed is a pilot run. For thesis-level evidence: ≥50 trials × 3 seeds (42, 123, 7).

4. **Blind baseline interpretation:** Shuffled labels create a noise floor, not a true never-trained baseline. Report excess MAD = real − blind, not raw MAD alone.

5. **Unlearning cost:** Report `Path B time − Path A time` as the overhead of a correct deletion. For in-context models this should be small.

## What comes next

| Step | Task |
|---|---|
| Next | Re-run with seeds 123 and 7; increase to 50 trials |
| Week 5–6 | Membership Inference Attack (MIA) probes |
| Week 7 | `fit_mode="fit_with_cache"` audit |
| Week 8 | Write-up |